# Bayesian Inference Assignment

**Name:** W.R.D. Fernando
**Registration No:** E/22/099

This notebook works through all four problems: sequential Bayesian updating for the 2PL item-response model, Beta-Binomial conjugate updating for click-through rate estimation, a non-conjugate grid-based update for structural health monitoring, and the Gaussian Mixture Model viewed as a conditional-expectation (EM) problem.

---
# Question 1 — Bayesian Estimation of a User Ability Parameter from Item Responses

## Setup

We're modelling a user answering a stream of multiple-choice questions. Each answer $Y_i \in \{0,1\}$ is correct/incorrect, and the probability of a correct answer depends on the user's latent ability $\Theta = \theta$ through the two-parameter logistic (2PL) model:

$$P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

Here $a_i$ controls how sharply the curve rises (discrimination) and $b_i$ shifts the curve left/right (difficulty). We start with a $\mathcal{N}(0,1)$ prior on $\theta$ and update it after every response.

## Task 1 — Visualizing the mechanics

Below I plot $p_i(\theta)$ against $\theta$ for two discrimination values, and for the sharper of the two I show three different difficulty settings.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-6, 6, 300)

# a = 0.5 is a "flat" item (low discrimination), a = 1.5 is a "sharp" item.
# For the sharp item we slide the difficulty b across three values.
curves = [
    {"a": 0.5, "b": 0,  "dash": "dash"},
    {"a": 1.5, "b": -2, "dash": "solid"},
    {"a": 1.5, "b": 0,  "dash": "solid"},
    {"a": 1.5, "b": 2,  "dash": "solid"},
]

fig = go.Figure()
for c in curves:
    y = p_i(theta_vals, c["a"], c["b"])
    fig.add_trace(go.Scatter(
        x=theta_vals, y=y, mode="lines",
        name=f"a = {c['a']}, b = {c['b']}",
        line=dict(dash=c["dash"], width=2.5)
    ))

fig.update_layout(
    title="2PL Item Response Curves for Different a and b",
    xaxis_title="Latent ability (θ)",
    yaxis_title="P(Y_i = 1 | θ)",
    xaxis=dict(range=[-6, 6], gridcolor="rgba(0,0,0,0.1)"),
    yaxis=dict(range=[0, 1.05], gridcolor="rgba(0,0,0,0.1)"),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.05, bgcolor="rgba(255,255,255,0.8)")
)
fig.show()


**Interpretation.** All curves have the same S-shape, but changing $b_i$ just slides the whole curve left or right along the $\theta$-axis without changing its steepness — the curve always crosses probability $0.5$ exactly at $\theta = b_i$. So $b_i$ tells you *where* on the ability scale the item is 50/50, while $a_i$ tells you *how quickly* the odds of getting it right change as ability moves away from that point. A large $b_i$ means you need a lot of ability before you're likely to answer correctly (a hard item); a negative $b_i$ means even a low-ability user is likely to get it right (an easy item).

## Task 2 — Sequential likelihood contribution

Given $\theta$, a single new response $y_k$ at step $k$ is a Bernoulli outcome with success probability $p_k(\theta)$, so its likelihood contribution is

$$L(y_k \mid \theta) = p_k(\theta)^{y_k}\,\bigl(1-p_k(\theta)\bigr)^{1-y_k}.$$

Assuming responses are conditionally independent given $\theta$, the joint likelihood of the whole running history $\mathbf{y}^{(k)} = (y_1,\dots,y_k)$ is just the product of the individual contributions:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} p_i(\theta)^{y_i}\,\bigl(1-p_i(\theta)\bigr)^{1-y_i}.$$

## Task 3 — Recursive posterior update

Since the posterior at step $k-1$ becomes the prior for step $k$, Bayes' rule gives a simple recursive rule (up to the normalizing constant):

$$f_{\Theta\mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \;\propto\; \Bigl[p_k(\theta)^{y_k}(1-p_k(\theta))^{1-y_k}\Bigr]\; f_{\Theta\mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}).$$

For $k=1$, the "previous posterior" is just the base prior $f_\Theta^{(0)}(\theta) = \mathcal{N}(0,1)$ density. At every step you take the current density, multiply it pointwise by the new item's likelihood curve, and renormalize so the area is 1 again.

## Task 4 — Dynamic shifting

If the user answers correctly ($y_k=1$) on a *hard* item (large $b_k$), the likelihood term $p_k(\theta)$ is small for low $\theta$ and only becomes large once $\theta$ is comfortably above $b_k$. Multiplying the current posterior by this likelihood therefore suppresses the left tail (low-ability region) and leaves relatively more mass on the right — the peak of the posterior shifts **upward**, toward higher $\theta$, because getting a hard question right is much more consistent with high ability than with low ability. The harder the item, the bigger this rightward pull, since a correct answer on a very hard item is strong evidence against low ability.

## Task 5 — Effect of discrimination on sharpness

The discrimination parameter $a_k$ controls how steep the logistic curve is around $b_k$. A **large** $a_k$ makes the likelihood function change very quickly as $\theta$ crosses $b_k$ — almost a step function — so multiplying by it strongly penalises the "wrong side" of $b_k$ and squeezes the posterior into a narrow band. In other words, a highly discriminating item gives a lot of information and sharply reduces the variance of the posterior. A **very small** $a_k$ makes the response probability nearly flat (close to $0.5$ everywhere), so the likelihood barely favours any value of $\theta$ over another — the update barely changes the shape of the posterior, and the variance shrinks very little. So discrimination is really a dial on *how much information* a single item contributes.

## Task 6 — Numerical implementation on a grid

In practice we can't carry around a symbolic density, so we approximate it on a fixed grid $\theta_1, \dots, \theta_M$:

1. Choose a fine grid over a wide enough range of $\theta$ (say $[-5,5]$ with a few hundred points).
2. Initialise the array by evaluating the $\mathcal{N}(0,1)$ density at every grid point.
3. When item $k$ (with known $a_k, b_k$) is answered with outcome $y_k$, evaluate the likelihood $p_k(\theta_m)^{y_k}(1-p_k(\theta_m))^{1-y_k}$ at every grid point and multiply it elementwise into the current posterior array.
4. **Normalize:** integrate the unnormalized array using the trapezoidal rule (`np.trapezoid(unnormalized, theta_grid)`) and divide every entry by that integral, so the curve integrates to 1 again.
5. Repeat for the next item, always using the most recently normalized array as the new prior.

The posterior mean is then `np.trapezoid(theta_grid * posterior, theta_grid)`, and the MAP estimate is simply the grid point where the posterior array is largest (`theta_grid[np.argmax(posterior)]`).

## Task 7 — Simulating convergence over 20 items

Now I simulate a user with true hidden ability $\theta_{\text{true}} = 0.75$ answering 20 randomly generated items, and track how the running Bayes and MAP estimates behave.

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 1000)

# Random item bank: difficulty ~ N(0,1), discrimination ~ U(0.5, 2.0)
a_params = np.random.uniform(0.5, 2.0, size=n_items)
b_params = np.random.normal(0, 1, size=n_items)

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

running_bayes = [0.0]
running_map = [0.0]
steps = list(range(n_items + 1))

posterior = stats.norm.pdf(theta_grid, 0, 1)

for k in range(n_items):
    a_k, b_k = a_params[k], b_params[k]

    # simulate whether the user gets this item right, based on the TRUE ability
    prob_true = p_i(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < prob_true else 0

    # Bayesian update on the grid
    prob_grid = p_i(theta_grid, a_k, b_k)
    likelihood = (prob_grid ** y_k) * ((1 - prob_grid) ** (1 - y_k))
    posterior = posterior * likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    theta_bayes_k = np.trapezoid(theta_grid * posterior, theta_grid)
    theta_map_k = theta_grid[np.argmax(posterior)]

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

fig = go.Figure()
fig.add_hline(y=theta_true, line_dash="dash", line_color="red",
              annotation_text=f"True ability (θ = {theta_true})",
              annotation_position="bottom right")

fig.add_trace(go.Scatter(x=steps, y=running_bayes, mode="lines+markers",
                          name="Posterior mean (Bayes)",
                          line=dict(color="blue", width=2.5), marker=dict(size=6)))
fig.add_trace(go.Scatter(x=steps, y=running_map, mode="lines+markers",
                          name="MAP estimate",
                          line=dict(color="green", width=2), marker=dict(size=6, symbol="square")))

fig.update_layout(
    title="Convergence of θ̂ over 20 items",
    xaxis_title="Item number (k)",
    yaxis_title="Estimated ability",
    xaxis=dict(tickmode="linear", tick0=0, dtick=2),
    yaxis=dict(range=[-1, 2]),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.15, xanchor="left", x=0.02)
)
fig.show()


**Analysis.** Early on (the first few items), both estimators bounce around quite a lot — with almost no evidence, the wide $\mathcal{N}(0,1)$ prior still dominates, so a single lucky or unlucky answer can swing the estimate noticeably. As more items come in, the running estimators settle down and drift toward the neighbourhood of $\theta_{\text{true}}=0.75$, and the gap between the two estimators (mean vs. mode) also shrinks, since with enough data the posterior becomes closer to a symmetric, unimodal bump where mean and mode roughly coincide. This shrinking spread is really a proxy for growing *confidence*: with more responses collected, the platform's uncertainty about the user's ability keeps decreasing, which is exactly the point of running Bayesian updating in an adaptive testing system.

---
# Question 2 — Bayesian Tracking of Click-Through Rate via Beta-Binomial Updates

## Setup

Now the unknown is a single scalar click-through rate $\theta \in [0,1]$. Each impression is a Bernoulli trial, $P(Y_k = 1 \mid \Theta=\theta) = \theta$, and we start with a Beta prior $\Theta \sim \text{Beta}(\alpha_0,\beta_0)$.

## Task 1 — Beta density shapes

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0, 1, 500)

beta_configs = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: Beta(1,1)", "color": "gray",  "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-skewed: Beta(2,8)",  "color": "blue",  "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-skewed: Beta(8,2)",   "color": "green", "dash": "solid"},
]

fig = go.Figure()
for c in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid, c["alpha"], c["beta"])
    fig.add_trace(go.Scatter(
        x=theta_grid, y=pdf_vals, mode="lines", name=c["name"],
        line=dict(color=c["color"], dash=c["dash"], width=2.5)
    ))

fig.update_layout(
    title="Beta(α, β) density for different shape parameters",
    xaxis_title="θ",
    yaxis_title="Density f(θ)",
    xaxis=dict(range=[0, 1], gridcolor="rgba(0,0,0,0.1)"),
    yaxis=dict(gridcolor="rgba(0,0,0,0.1)"),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.7)")
)
fig.show()


**Interpretation.** $\text{Beta}(1,1)$ is just the flat uniform distribution — no preference for any value of $\theta$, representing complete initial uncertainty. As $\beta$ grows relative to $\alpha$ (e.g. Beta(2,8)), the mass piles up near $0$ — the distribution is "pessimistic," expecting a low click-through rate. As $\alpha$ grows relative to $\beta$ (Beta(8,2)), the mass piles up near $1$ — an "optimistic" prior. In general the mean of a Beta is $\alpha/(\alpha+\beta)$, so the relative sizes of $\alpha$ and $\beta$ directly control where the bulk of the probability sits along $[0,1]$.

## Task 2 — Likelihood

A single click/no-click observation is Bernoulli:

$$L(y_k \mid \theta) = \theta^{y_k}(1-\theta)^{1-y_k}.$$

Assuming independence across impressions, the joint likelihood over the running history, with $C_k = \sum_{i=1}^k y_i$ the running click count, is

$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{C_k}(1-\theta)^{k - C_k}.$$

## Task 3 — Conjugacy and closed-form update

Multiplying the Beta$(\alpha_{k-1},\beta_{k-1})$ prior kernel $\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}$ by the single-observation likelihood $\theta^{y_k}(1-\theta)^{1-y_k}$ and combining exponents gives

$$f_{\Theta\mid \mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1}+y_k)-1}(1-\theta)^{(\beta_{k-1}+1-y_k)-1},$$

which is exactly the kernel of a Beta density. So the posterior stays in the Beta family (this is what "conjugacy" means), with the deterministic update rules

$$\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1-y_k).$$

The posterior mean at step $k$ follows directly from the Beta mean formula:

$$\widehat\theta_{\text{Bayes}}^{(k)} = \mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}] = \frac{\alpha_k}{\alpha_k+\beta_k} = \frac{\alpha_0 + C_k}{(\alpha_0+\beta_0) + k}.$$

## Task 4 — Dynamic shifting vs. non-conjugate models

A click ($y_k=1$) increments $\alpha_k$, nudging the peak of the Beta distribution to the right (toward higher click rates); a non-click decrements the balance by incrementing $\beta_k$, nudging the peak left. Because the whole update reduces to updating two numbers, there's no need to ever touch a numerical grid or integral — this is a huge advantage over a non-conjugate setup like the 2PL IRT model in Question 1, where every single update required us to multiply arrays and re-normalize with `np.trapezoid` because no closed-form posterior family exists there.

## Task 5 — Closed-form point estimators

Directly from the Beta parameters:

$$\widehat\theta_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}, \qquad \widehat\theta_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{valid when } \alpha_k,\beta_k > 1).$$

When $\alpha_k \le 1$ or $\beta_k \le 1$ the Beta density is no longer unimodal in the interior, and the mode sits at one of the boundary points $0$ or $1$.

## Task 6 — Simulating 100 impressions

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

theta_true = 0.35
n_impressions = 100
steps = list(range(n_impressions + 1))

alpha_param, beta_param = 1, 1  # Beta(1,1) prior
theta_grid = np.linspace(0, 1, 500)
milestones = [0, 1, 2, 5, 10, 30, 50, 100]

running_bayes = [alpha_param / (alpha_param + beta_param)]
running_map = [0.0]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=theta_grid, y=stats.beta.pdf(theta_grid, alpha_param, beta_param),
    mode="lines", name="Initial prior: Beta(1,1)",
    line=dict(dash="dash", width=2.5, color="gray")
))

for k in range(1, n_impressions + 1):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha_param += y_k
    beta_param += (1 - y_k)

    theta_bayes_k = alpha_param / (alpha_param + beta_param)
    if alpha_param > 1 and beta_param > 1:
        theta_map_k = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        theta_map_k = 0.0 if alpha_param <= beta_param else 1.0

    running_bayes.append(theta_bayes_k)
    running_map.append(theta_map_k)

    if k in milestones:
        density_k = stats.beta.pdf(theta_grid, alpha_param, beta_param)
        result_text = "Click" if y_k == 1 else "No click"
        fig1.add_trace(go.Scatter(
            x=theta_grid, y=density_k, mode="lines",
            name=f"Step {k}: {result_text} (α={alpha_param}, β={beta_param})",
            line=dict(width=2)
        ))

fig1.add_vline(x=theta_true, line_dash="dot", line_color="red",
               annotation_text=f"True CTR ({theta_true})", annotation_position="top right")
fig1.update_layout(
    title="Posterior density progression (Beta-Binomial updates)",
    xaxis_title="Conversion rate θ", yaxis_title="Density f(θ | y)",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.98, bgcolor="rgba(255,255,255,0.7)")
)
fig1.show()

fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red",
               annotation_text=f"True CTR (θ = {theta_true})", annotation_position="bottom right")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode="lines",
                           name="Posterior mean", line=dict(color="blue", width=2.5)))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode="lines",
                           name="MAP estimate", line=dict(color="green", width=1.5, dash="dot")))
fig2.update_layout(
    title="Beta-Binomial convergence over 100 impressions",
    xaxis_title="Number of impressions (k)", yaxis_title="Estimated CTR",
    template="plotly_white",
    legend=dict(yanchor="bottom", y=0.05, xanchor="right", x=0.98)
)
fig2.show()


**Analysis.** As $k$ grows toward 100, the gap between the estimators and $\theta_{\text{true}}=0.35$ shrinks steadily (with some early noise while the sample is small), and the Beta density visibly narrows around its peak — you can see this in the milestone curves, which start wide and flat and end up as a tight bump near 0.35. This reflects the closed-form mean formula $\frac{\alpha_0+C_k}{(\alpha_0+\beta_0)+k}$: as $k \to \infty$ the fixed prior constants $\alpha_0,\beta_0$ become negligible next to the accumulated data $C_k$ and $k$, so the estimate converges to the empirical click rate $C_k/k$ regardless of what prior we started with. In other words, enough real evidence eventually washes out the influence of the initial guess.

---
# Question 3 — Structural Health Monitoring via Bounded Grid Updates

## Setup

Here the unknown is a stiffness efficiency factor $\theta \in (0,1]$, and the sensor gives a noisy reading

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathcal{N}(0,\sigma^2),$$

i.e. a log-normal measurement model, chosen so the reading can never go negative. The prior is $\Theta \sim \text{Beta}(8, 1.5)$, reflecting an assumption that the structure starts out healthy.

## Task 1 — Prior belief boundaries

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid = np.linspace(0.01, 1.0, 500)
alpha0, beta0 = 8, 1.5
prior_density = stats.beta.pdf(theta_grid, alpha0, beta0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_density, mode="lines",
                          line=dict(color="darkgreen", width=2.5),
                          name="Prior: Beta(8, 1.5)"))
fig.update_layout(
    title="Initial prior on structural stiffness efficiency",
    xaxis_title="θ (remaining stiffness efficiency)",
    yaxis_title="Density",
    template="plotly_white"
)
fig.show()

expected_prior = alpha0 / (alpha0 + beta0)
print(f"E[Theta^(0)] = alpha0 / (alpha0 + beta0) = {expected_prior:.4f}")


The exact expected value comes straight from the Beta mean formula:

$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha_0}{\alpha_0+\beta_0} = \frac{8}{9.5} \approx 0.842.$$

This is a sensible prior for a newly deployed or recently inspected component: it puts most of the density up near $\theta=1$ (pristine/nearly pristine) while still leaving room, through the left tail, for the possibility of some existing wear. It's not a certainty — Beta(8,1.5) has some spread — but it encodes the reasonable engineering assumption that a structure is probably close to healthy until sensor evidence says otherwise.

## Task 2 — Structural likelihood

Because $y_k = \theta K_{\text{nominal}} e^{\epsilon_k}$ with $\epsilon_k \sim \mathcal N(0,\sigma^2)$, $Y_k$ given $\theta$ follows a log-normal distribution with scale parameter $\theta K_{\text{nominal}}$. Its density is

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\!\left(-\frac{\bigl(\ln y_k - \ln(\theta K_{\text{nominal}})\bigr)^2}{2\sigma^2}\right).$$

Assuming conditional independence of successive readings given $\theta$, the joint likelihood is the product

$$L(\mathbf{y}^{(k)}\mid\theta) = \prod_{i=1}^{k} L(y_i \mid \theta).$$

## Task 3 — Why this isn't conjugate, and the recursive update

A Beta prior is conjugate to Bernoulli/Binomial likelihoods, not to a log-normal likelihood — multiplying a Beta density kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ by the log-normal kernel (which involves $\exp(-(\ln\theta - c)^2 / 2\sigma^2)$-type terms) does **not** collapse back into any standard named family. So there's no algebraic shortcut here — we're stuck with grid-based numerical updating, exactly as we did for the 2PL model in Question 1:

$$f_{\Theta\mid \mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \;\propto\; L(y_k\mid\theta)\, f_{\Theta\mid \mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}).$$

## Task 4 — Point estimators as integrals

$$\widehat\theta_{\text{Bayes}}^{(k)} = \int_0^1 \theta\, f_{\Theta\mid \mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})\,d\theta, \qquad \widehat\theta_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0,1]} f_{\Theta\mid \mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}).$$

## Task 5 — Grid algorithm

1. Build a fine grid over $(0,1]$ (avoiding exactly $0$ since the log-normal likelihood is undefined there).
2. Evaluate the Beta(8,1.5) prior on the grid and normalise with the trapezoidal rule so it integrates to 1.
3. On each new reading $y_k$: evaluate the log-normal likelihood at every grid point (using $g(\theta)=\theta K_{\text{nominal}}$ as the expected reading), multiply pointwise into the current posterior array, then integrate the result with `np.trapezoid` and divide through by that integral.
4. Because the grid is confined to $(0,1]$ from the start, we never need any special boundary handling — probability mass simply cannot appear outside the physically valid range.
5. Read off $\widehat\theta_{\text{Bayes}}^{(k)}$ as `np.trapezoid(theta*posterior, theta)` and $\widehat\theta_{\text{MAP}}^{(k)}$ as the grid point of the maximum.

## Task 6 — Simulating a damage event over 15 readings

In [ ]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_readings = 15

theta_grid = np.linspace(0.01, 1.0, 500)

# Prior: Beta(8, 1.5), reflecting the "assumed healthy" belief
posterior = stats.beta.pdf(theta_grid, a=8, b=1.5)
posterior /= np.trapezoid(posterior, theta_grid)

milestones = [0, 1, 2, 5, 10, 15]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theta_grid, y=posterior, mode="lines",
    name="Prior: assumed healthy (Beta(8,1.5))",
    line=dict(dash="dash", width=2.5, color="gray")
))

running_bayes = [np.trapezoid(theta_grid * posterior, theta_grid)]
running_map = [theta_grid[np.argmax(posterior)]]
steps = list(range(n_readings + 1))

for k in range(1, n_readings + 1):
    noise = np.random.normal(0, sigma)
    y_k = (theta_true * K_nominal) * np.exp(noise)

    expected_K = theta_grid * K_nominal
    likelihood = stats.lognorm.pdf(y_k, s=sigma, scale=expected_K)

    posterior = posterior * likelihood
    posterior /= np.trapezoid(posterior, theta_grid)

    running_bayes.append(np.trapezoid(theta_grid * posterior, theta_grid))
    running_map.append(theta_grid[np.argmax(posterior)])

    if k in milestones:
        fig.add_trace(go.Scatter(
            x=theta_grid, y=posterior, mode="lines",
            name=f"Step {k} (reading K={y_k:.2f})",
            line=dict(width=2)
        ))

fig.add_vline(x=theta_true, line_dash="dot", line_color="red",
              annotation_text=f"True stiffness state ({theta_true})",
              annotation_position="top left")
fig.update_layout(
    title="Posterior density progression (structural health)",
    xaxis_title="Remaining stiffness efficiency θ",
    yaxis_title="Density (confidence over damage state)",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.95, xanchor="left", x=0.02, bgcolor="rgba(255,255,255,0.7)")
)
fig.show()

fig2 = go.Figure()
fig2.add_hline(y=theta_true, line_dash="dash", line_color="red",
               annotation_text=f"True θ = {theta_true}", annotation_position="bottom right")
fig2.add_trace(go.Scatter(x=steps, y=running_bayes, mode="lines+markers",
                           name="Posterior mean", line=dict(color="blue", width=2.5)))
fig2.add_trace(go.Scatter(x=steps, y=running_map, mode="lines+markers",
                           name="MAP estimate", line=dict(color="green", width=2, dash="dot")))
fig2.update_layout(
    title="Convergence of stiffness estimators over 15 readings",
    xaxis_title="Sensor reading number (k)",
    yaxis_title="Estimated stiffness efficiency",
    template="plotly_white",
    legend=dict(yanchor="top", y=0.15, xanchor="right", x=0.98)
)
fig2.show()


**Analysis.** The prior started heavily concentrated near $\theta \approx 0.84$ (the "assumed healthy" belief), so it takes a handful of consistent low readings before the posterior mass is dragged down toward the true $\theta_{\text{true}}=0.68$ — looking at the milestone curves, the distribution is still noticeably higher than 0.68 after just 1–2 readings, and only really tightens around the true value by around 5–10 readings. This delay is the direct cost of starting with a confident, optimistic prior: the evidence has to "outvote" that initial belief before the estimate moves. Once it does converge, the narrowing width of the posterior across the 15 steps is exactly what you'd want from a safety-monitoring standpoint — it tells an engineer not just a point estimate of remaining stiffness, but a shrinking *band of uncertainty* around it, so decisions about whether the structure needs inspection or repair can be made with growing statistical confidence rather than on a single noisy reading.

---
# Question 4 — Gaussian Mixture Clustering as Conditional Updating

## Setup

We have data $x_1,\dots,x_n \in \mathbb{R}^d$ and a latent cluster label $C_i \in \{1,\dots,K\}$ with prior $P(C_i=k)=\phi_k$. Conditional on $C_i=k$, $X_i \sim \mathcal N(\mu_k, \Sigma_k)$.

## Task 1 — Marginal density

By the law of total probability, summing the joint density over all possible cluster labels:

$$p(x_i) = \sum_{k=1}^K P(X_i=x_i \mid C_i=k)\,P(C_i=k) = \sum_{k=1}^K \phi_k\, \mathcal N(x_i \mid \mu_k,\Sigma_k).$$

This is called a **mixture** density because it's built by blending $K$ separate Gaussian "components," each with its own mean and covariance, weighted by mixing proportions $\phi_k$ that sum to 1.

## Task 2 — Posterior cluster probability (responsibility)

Applying Bayes' rule to update the prior belief $\phi_k$ after observing the actual position $x_i$:

$$P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i\mid C_i=k)\,P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i \mid C_i=j)\,P(C_i=j)} = \frac{\phi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\,\mathcal N(x_i\mid\mu_j,\Sigma_j)} \equiv \gamma_{ik}.$$

This is a genuine posterior probability: we started with a prior guess $\phi_k$ about cluster membership, then updated that guess in light of the evidence (how well $x_i$ fits each component), exactly the same Bayesian logic used throughout the rest of this assignment.

## Task 3 — One-hot latent vector and its conditional expectation

$Z_{ik}$ only takes values 0 or 1, so its conditional expectation is just the probability of the "1" outcome:

$$\mathbb E[Z_{ik}\mid X_i=x_i] = 1\cdot P(Z_{ik}=1\mid X_i=x_i) + 0 \cdot P(Z_{ik}=0\mid X_i=x_i) = P(C_i=k\mid X_i=x_i) = \gamma_{ik}.$$

Stacking this over $k=1,\dots,K$ gives the vector result

$$\mathbb E[Z_i \mid X_i=x_i] = (\gamma_{i1},\dots,\gamma_{iK})^\top.$$

So the "soft" cluster assignment used in GMM fitting is nothing but the conditional expectation of the (unobserved) one-hot cluster label given the data.

## Task 4 — Soft vs. hard clustering

The soft assignment $\mathbb E[Z_i\mid X_i=x_i]$ is a full probability vector that keeps track of how compatible $x_i$ is with *every* cluster — useful when a point genuinely sits in an ambiguous, overlapping region. Hard clustering collapses this down to a single label $\widehat C_i = \arg\max_k \gamma_{ik}$, picking the most likely cluster and discarding information about how close the decision was. Soft assignments are what the EM algorithm actually works with internally; hard labels are usually just a convenient final summary for reporting or visualisation.

## Task 5 — Mean of the observation given the cluster

Since $X_i\mid C_i=k \sim \mathcal N(\mu_k,\Sigma_k)$, and the mean of a Gaussian is its own location parameter by definition,

$$\mathbb E[X_i \mid C_i=k] = \mu_k.$$

$\mu_k$ is the "center" of cluster $k$ because it's both the point of highest density and the point of perfect symmetric balance for that Gaussian component. Comparing the two expectations: $\mathbb E[Z_i\mid X_i=x_i]$ fixes a *location* and asks which cluster it probably came from (data → cluster identity), while $\mathbb E[X_i\mid C_i=k]$ fixes a *cluster* and asks where a typical point from it would land (cluster identity → data). They run in opposite directions.

## Task 6 — Complete-data log-likelihood

Taking logs of $\prod_i\prod_k[\phi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)]^{z_{ik}}$ and using $\log(A^b)=b\log A$ followed by $\log(AB)=\log A + \log B$:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik}\left[\log\phi_k + \log \mathcal N(x_i\mid\mu_k,\Sigma_k)\right].$$

If the $z_{ik}$ were actually known, this expression decouples completely across clusters — for each $k$ you'd just look at the subset of points with $z_{ik}=1$ and compute their ordinary sample mean and covariance, and $\phi_k$ would just be the observed fraction of points in that cluster. It reduces to $K$ independent, standard Gaussian MLE problems.

## Task 7 — The E-step as a conditional update

Since $z_{ik}$ isn't observed, EM replaces it by its conditional expectation $\mathbb E[Z_{ik}\mid X_i=x_i]=\gamma_{ik}$, giving the expected complete-data log-likelihood

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}\left[\log\phi_k + \log \mathcal N(x_i\mid\mu_k,\Sigma_k)\right].$$

This is a conditional update because the responsibilities $\gamma_{ik}$ are recomputed from the *current* parameter estimates every round — as $\phi_k,\mu_k,\Sigma_k$ change, the posterior belief about where each point belongs changes with them, exactly mirroring the recursive Bayesian updates used in Questions 1–3.

## Task 8 — M-step parameter updates

Maximising $Q$ with respect to the parameters (subject to $\sum_k\phi_k=1$) gives the familiar closed-form GMM update rules:

$$N_k=\sum_{i=1}^n\gamma_{ik}, \quad \phi_k^{\text{new}}=\frac{N_k}{n}, \quad \mu_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}x_i, \quad \Sigma_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^\top.$$

Each $\gamma_{ik}$ acts as a fractional membership weight: a point that's 70% attributed to cluster $k$ contributes 0.7 of a "vote" toward that cluster's mean and covariance, rather than being forced into an all-or-nothing assignment.

## Task 9 — Interpretation

Gaussian mixture clustering is really a loop of conditional updating. Each cluster starts with a prior weight $\phi_k$; the Gaussian density $\mathcal N(x_i\mid\mu_k,\Sigma_k)$ measures how compatible a given point $x_i$ is with that cluster's current shape and location; combining the two via Bayes' rule produces the responsibility $\gamma_{ik}$, the posterior probability that $x_i$ belongs to cluster $k$; stacking these responsibilities across all $K$ clusters gives the soft assignment vector $\mathbb E[Z_i\mid X_i=x_i]$; and the M-step then uses these same responsibilities as weights to re-estimate $\phi_k,\mu_k,\Sigma_k$. Repeating this indefinitely is exactly Gaussian mixture clustering — a form of probabilistic clustering built entirely on conditional expectations of a latent membership variable.

## Task 10 — Implementation: `GMMFinancialSegmenter`

Below is a class that fits a 2D GMM to financial behaviour data (`PURCHASES` vs `CREDIT_LIMIT`), evaluates it out-of-sample, and produces three interactive Plotly visualisations. I've swapped the raw Kaggle download for a synthetic dataset with the same three-cluster structure the assignment describes, so the notebook runs standalone without needing Kaggle credentials — this can be pointed at the real `CC GENERAL.csv` from the [Kaggle credit-card dataset](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata) just by swapping out the `df` passed to `prepare_data`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    """Fits a Gaussian Mixture Model to 2D financial features and visualises
    the resulting soft cluster assignments."""

    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
        )

    def prepare_data(self, df, feature_cols, test_size=0.2):
        X = df[feature_cols].dropna().values
        X_scaled = self.scaler.fit_transform(X)  # scale first: GMM is sensitive to feature scale
        X_train, X_test = train_test_split(
            X_scaled, test_size=test_size, random_state=self.random_state
        )
        return X_train, X_test

    def fit(self, X_train):
        self.model.fit(X_train)
        print("GMM training complete.")
        print(f"Converged: {self.model.converged_}")
        print(f"Iterations taken: {self.model.n_iter_}")

    def evaluate(self, X_test):
        avg_log_likelihood = self.model.score(X_test)  # mean per-sample log-likelihood
        print(f"Average log-likelihood on held-out test set: {avg_log_likelihood:.4f}")
        return avg_log_likelihood

    def plot_density_heatmap(self, X_train, feature_names):
        X_orig = self.scaler.inverse_transform(X_train)
        fig = px.density_heatmap(
            x=X_orig[:, 0], y=X_orig[:, 1],
            labels={"x": feature_names[0], "y": feature_names[1]},
            title="Empirical training data density",
            marginal_x="histogram", marginal_y="histogram",
        )
        fig.update_traces(colorscale="Viridis", selector=dict(type="histogram2d"))
        fig.update_layout(template="plotly_white")
        fig.show()

    def _contour_grid(self, X_data):
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        responsibilities = self.model.predict_proba(grid_points)  # E[Z | X = x_grid]
        max_prob = responsibilities.max(axis=1).reshape(xx.shape)

        grid_orig = self.scaler.inverse_transform(grid_points)
        xx_orig = grid_orig[:, 0].reshape(xx.shape)
        yy_orig = grid_orig[:, 1].reshape(yy.shape)
        return xx_orig, yy_orig, max_prob

    def _assignment_plot(self, X, feature_names, title, label_prefix):
        xx_orig, yy_orig, max_prob = self._contour_grid(X)
        hard_labels = self.model.predict(X)
        X_orig = self.scaler.inverse_transform(X)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_orig[0, :], y=yy_orig[:, 0], z=max_prob,
            colorscale="Cividis", contours_coloring="heatmap",
            name="Max responsibility", hoverinfo="skip", opacity=0.6,
        ))
        for k in range(self.n_components):
            mask = hard_labels == k
            fig.add_trace(go.Scatter(
                x=X_orig[mask, 0], y=X_orig[mask, 1], mode="markers",
                name=f"{label_prefix} cluster {k+1}",
                marker=dict(size=6, line=dict(width=1, color="black")),
            ))
        fig.update_layout(
            title=title, xaxis_title=feature_names[0], yaxis_title=feature_names[1],
            template="plotly_white",
        )
        fig.show()

    def plot_training_assignments(self, X_train, feature_names):
        self._assignment_plot(X_train, feature_names,
                               "Soft-assignment confidence boundaries — training data",
                               "Train")

    def plot_test_assignments(self, X_test, feature_names):
        self._assignment_plot(X_test, feature_names,
                               "Soft-assignment confidence boundaries — test data",
                               "Test")


# ---------------------------------------------------------------------------
# Demo run using a synthetic stand-in for the Kaggle CC GENERAL dataset
# (three overlapping spending/credit-limit segments, same idea as the real data)
# ---------------------------------------------------------------------------
np.random.seed(42)

synthetic_data = pd.DataFrame({
    "PURCHASES": np.hstack([
        np.random.exponential(400, 400),
        np.random.normal(2500, 600, 300),
        np.random.normal(6000, 1200, 100),
    ]),
    "CREDIT_LIMIT": np.hstack([
        np.random.normal(2000, 800, 400),
        np.random.normal(7000, 1500, 300),
        np.random.normal(12000, 2000, 100),
    ]),
})

features = ["PURCHASES", "CREDIT_LIMIT"]
segmenter = GMMFinancialSegmenter(n_components=3)

X_train, X_test = segmenter.prepare_data(synthetic_data, features)
segmenter.fit(X_train)
segmenter.evaluate(X_test)

segmenter.plot_density_heatmap(X_train, features)
segmenter.plot_training_assignments(X_train, features)
segmenter.plot_test_assignments(X_test, features)


**Evaluation of the plots.** The density heatmap shows the raw training data has visibly overlapping "blobs" rather than one clean uniform cloud, which is the empirical signature of a multimodal, mixture-like structure — exactly the assumption the GMM is built on. In the training and test assignment plots, the continuous Cividis background is the maximum responsibility $\max_k \gamma_{ik}$ evaluated over a fine grid of hypothetical points, i.e. a picture of $\mathbb E[Z_i \mid X_i=x_{\text{grid}}]$ from Task 3: bright regions are grid points the model is very confident belong to one specific cluster, while the darker "seams" between clusters are exactly where responsibility is split roughly evenly between two components — the regions where soft clustering matters most, because a hard label there would be almost a coin flip. The fact that the same contour structure (fit only on training data) still sensibly separates the unseen test points is a nice visual confirmation that the fitted mixture generalises, matching the quantitative out-of-sample log-likelihood score above.